# Diferencia observada, ATE, ATT, ATC, CATE

### Descripción de Conceptos de Inferencia Causal

A continuación se definen los conceptos de inferencia causal aplicados en este análisis:

1. **Diferencia Observada (Observed Difference)**:
   * **Qué es**: Es la diferencia simple entre el promedio del resultado ($Y$) del grupo que recibió el tratamiento ($T=1$) y el promedio del grupo de control ($T=0$).
   * **Limitación**: No es una estimación causal válida si hay sesgo de selección (es decir, si los grupos de tratamiento y control difieren de forma sistemática en otras variables que también afectan a la supervivencia).

2. **ATE (Average Treatment Effect - Efecto Promedio del Tratamiento)**:
   * **Qué es**: Es el impacto promedio que tendría el tratamiento si se aplicara a **toda la población** bajo estudio.
   * **Cálculo**: Es el promedio de la diferencia entre el resultado potencial con tratamiento y sin tratamiento para cada individuo: $\mathbb{E}[Y(1) - Y(0)]$.

3. **ATT (Average Treatment Effect on the Treated - Efecto Promedio en los Tratados)**:
   * **Qué es**: Es el impacto promedio del tratamiento únicamente en aquellos individuos que **efectivamente recibieron el tratamiento** ($T=1$).
   * **Cálculo**: $\mathbb{E}[Y(1) - Y(0) \mid T=1]$. Responde a la pregunta: *¿Cuánto se beneficiaron de viajar en Primera Clase aquellos que realmente lo hicieron?*

4. **ATC (Average Treatment Effect on the Control - Efecto Promedio en los Controles)**:
   * **Qué es**: Es el impacto promedio que habría tenido el tratamiento únicamente en aquellos que **no lo recibieron** ($T=0$).
   * **Cálculo**: $\mathbb{E}[Y(1) - Y(0) \mid T=0]$. Responde a la pregunta: *¿Cuánto habrían mejorado sus probabilidades de sobrevivir los pasajeros de clases inferiores si hubieran viajado en Primera Clase?*

5. **CATE (Conditional Average Treatment Effect - Efecto Promedio Condicional del Tratamiento)**:
   * **Qué es**: Es el efecto promedio del tratamiento condicionado a ciertas características o subgrupos de la población (como género o grupos de edad).
   * **Cálculo**: $\mathbb{E}[Y(1) - Y(0) \mid X=x]$. Permite identificar la heterogeneidad del efecto (por ejemplo, ver si el tratamiento es más efectivo en hombres que en mujeres, o en jóvenes que en adultos mayores).

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

df = sns.load_dataset("titanic")

df = df[["survived", "pclass", "sex", "age", "sibsp", "parch"]].copy()

df["treatment"] = (df["pclass"] == 1).astype(int)

df = df.dropna().copy()

X = df[["sex", "age", "sibsp", "parch"]]
T = df["treatment"]
Y = df["survived"]

## Diferencia observada

In [2]:
observed_difference = (
    df.loc[T == 1, "survived"].mean()
    - df.loc[T == 0, "survived"].mean()
)

print(f"Observed difference: {observed_difference:.3f}")

Observed difference: 0.338


## ATE

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

features = ["treatment", "sex", "age", "sibsp", "parch"]
categorical = ["sex"]
numeric = ["treatment", "age", "sibsp", "parch"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", StandardScaler(), numeric),
])

model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

model.fit(df[features], Y)

df_treated = df[features].copy()
df_control = df[features].copy()

df_treated["treatment"] = 1
df_control["treatment"] = 0

pred_treated = model.predict_proba(df_treated)[:, 1]
pred_control = model.predict_proba(df_control)[:, 1]

ate = np.mean(pred_treated - pred_control)

print(f"Estimated ATE: {ate:.3f}")

Estimated ATE: 0.349


## ATT y ATC

In [4]:
att = np.mean(
    pred_treated[T.to_numpy() == 1]
    - pred_control[T.to_numpy() == 1]
)

atc = np.mean(
    pred_treated[T.to_numpy() == 0]
    - pred_control[T.to_numpy() == 0]
)

print(f"ATE: {ate:.3f}")
print(f"ATT: {att:.3f}")
print(f"ATC: {atc:.3f}")

ATE: 0.349
ATT: 0.325
ATC: 0.358


## CATE

In [5]:
results = df[["sex", "age"]].copy()
results["effect"] = pred_treated - pred_control

results["age_group"] = pd.cut(
    results["age"],
    bins=[0, 18, 40, 60, np.inf],
    labels=["0-18", "19-40", "41-60", "60+"],
    include_lowest=True
)

cate_by_sex = results.groupby("sex", observed=True)["effect"].mean()
cate_by_age = results.groupby("age_group", observed=True)["effect"].mean()

print(cate_by_sex)
print(cate_by_age)

sex
female    0.281123
male      0.388675
Name: effect, dtype: float64
age_group
0-18     0.319182
19-40    0.369463
41-60    0.334480
60+      0.238231
Name: effect, dtype: float64
